In [5]:
import json
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [6]:
OLLAMA_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_URL,api_key="ollama")

In [7]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [11]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format = {"type":"json_object"})
    
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'About page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Careers/Jobs', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Blog/Newest posts', 'url': 'https://edwarddonner.com/posts/'}]}

In [13]:
MODEL = "llama3.2"

In [14]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2


Found 5 relevant links


{'links': [{'type': 'About page', 'url': '//huggingface.co/'},
  {'type': 'Company page', 'url': '//huggingface.co/'},
  {'type': 'Jobs/Careers page', 'url': '//apply.workable.com/huggingface/'},
  {'type': 'Blog', 'url': '//blog.huggingface.co/'},
  {'type': 'GitHub', 'url': '//github.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to llama3.2

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
    # Skip empty URLs
        if not link.get("url"):
            continue
        result += f"\n\n### Link: {link['type']}\n"
        try:
            result += fetch_website_contents(link["url"])
        except Exception as e:
            result += f"\nCould not fetch page: {e}\n"
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
2 days ago
•
158k
•
564
Zyphra/ZAYA1-8B
Updated
2 days ago
•
66.1k
•
391
deepseek-ai/DeepSeek-V4-Pro
Updated
5 days ago
•
2.02M
•
3.83k
HiDream-ai/HiDream-O1-Image
Updated
1 day ago
•
3.42k
•
199
google/gemma-4-31B-it-assistant
Updated
35 minutes ago
•
66.6k
•
200
Browse 2M+ models
Spaces
Paused
MCP
441
Qwen Image Edit + Loras built-in
👀
441
Qwen image edit with 🔞 loras
Running
on
Zero
MCP
1.07k
Wan2.2 14B Fast Preview
🐌
1.07k
generate a video from an image with a text prompt
Running
122
The ultimate g

In [18]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nSulphurAI/Sulphur-2-base\nUpdated\n2 days ago\n•\n158k\n•\n565\nZyphra/ZAYA1-8B\nUpdated\n2 days ago\n•\n66.1k\n•\n391\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n5 days ago\n•\n2.02M\n•\n3.83k\nHiDream-ai/HiDream-O1-Image\nUpdated\n1 day ago\n•\n3.42k\n•\n199\ngoogle/gemma-4-31B-it-assistant\nUpdated\n36 minutes ago\n•\n66.6k\n•\n200\nBrowse 2M+ models\nSpaces\nPaused\nMCP\n441\nQwen Image E

In [21]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links


# Hugging Face: Empowering the Future of AI Collaboration

At Hugging Face, we are a community-driven platform dedicated to democratizing access to artificial intelligence (AI) and transforming industries through collaborative innovation. With our cutting-edge platform, machine learning professionals, researchers, and developers can unleash their full potential, drive breakthroughs, and shape the future of technology.

## Our Vision

Our vision is to become the go-to hub for AI innovation, fostering a global community that brings together diverse expertise and perspectives to tackle complex challenges. We aim to accelerate the development of intelligent systems by providing an open-source, collaborative platform that enables seamless interactions between humans, models, and data.

## Our Mission

Our mission is to empower individuals to unlock their full potential in machine learning, making it more accessible, scalable, and effective for the greater good. We achieve this through:

* Hosting unlimited public models, datasets, and applications
* Providing an open-source stack that accelerates AI development
* Offering cutting-edge AI modalities across text, image, video, audio, and 3D domains
* Promoting a culture of collaboration, sharing, and community engagement

## Our Community

Our vibrant community is comprised of visionary researchers, industry leaders, startups, and individuals who share our passion for harnessing the power of AI. They collaborate on projects, share knowledge, and drive innovation towards meaningful impact.

## Careers & Jobs

At Hugging Face, we are committed to fostering a diverse and inclusive work environment that attracts top talent from around the world. Whether you're a researcher, developer, project manager, or sales professional, we offer exciting opportunities for career growth and contribution to our mission.

* Explore our job openings at [link]
* Join our community and get involved in AI projects and discussions on GitHub, Discord, or Medium

## Supporting a Better Future

We believe that the future of technology should be designed with humanity in mind. Our platform is poised to revolutionize industries such as healthcare, education, transportation, and more by putting AI empowerment into the hands of innovators worldwide.

Join us on this transformative journey and let's build a better future together!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 5 relevant links


**Hugging Face: Empowering the Machine Learning Community**

At Hugging Face, we believe that machine learning has the potential to revolutionize the way we live and work. Our mission is to build a platform where the machine learning community can collaborate, share knowledge, and accelerate innovation.

**Mission and Vision**

Our goal is to create an open-source ecosystem for machine learning, enabling researchers, developers, and organizations to build, deploy, and scale AI applications quickly and efficiently. We're committed to making AI more accessible, user-friendly, and accountable.

**Company Culture**

We're passionate about creating a collaborative and inclusive environment that fosters innovation, creativity, and growth. Our team values:

* **Community-driven**: We believe that the machine learning community is our greatest asset.
* **Open-source**: We're committed to making our technology available to everyone, free from restrictions.
* **Innovation**: We encourage experimentation, exploration, and pushing boundaries.

**Target Audience**

We serve a diverse range of customers, including:

* **Researchers and scientists**: Those exploring the frontiers of machine learning and artificial intelligence.
* **Developers and engineers**: Anyone building AI-powered applications or integrating machine learning into their workflows.
* **Entrepreneurs and startups**: Businesses leveraging machine learning to drive growth, efficiency, and innovation.

**Features and Applications**

Our platform offers a wide range of features and applications, including:

* **Model Hub**: Access over 2 million pre-trained models for various NLP, computer vision, and other applications.
* **Collaboration Tools**: Host, share, and integrate models, datasets, and applications with ease.
* **Agents**: Leverage our high-quality voice cloning technology (OmniVoice) to create human-like voice assistants.
* **Datasets**: Discover and leverage 500k+ open-source datasets for various machine learning tasks.

**News and Trends**

Stay up-to-date with the latest news and trends in machine learning on our blog, where we share insights on:

* **The latest model updates**
* **Emerging applications of AI**
* **Advances in natural language processing (NLP)**

Join us at Hugging Face, and experience the power of open-source collaboration that's shaping the future of machine learning.

In [25]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links


Hugging Face: Where Machine Learning Dreams Come True
=====================================================

Welcome to the most epic AI party on Earth! Hugging Face is where the machine learning community comes together to create, collaborate, and accelerate.

**Our Mission**

We're on a quest to make sense of the world, one model at a time. Our platform allows you to host, collaborate, and explore unlimited public models, datasets, and applications. Move faster with our open-source stack, and build your portfolio while sharing your work with the world.

**What We're All About**

We're passionate about making machine learning more accessible and fun for everyone. Our collaboration platform is designed to help you create, discover, and learn from the best models in AI.

*   **2M+ Models**: Browse our vast library of pre-trained models, each pushing the boundaries of what's possible with AI.
*   **500k+ Datasets**: Unlock hidden insights and patterns in our extensive collection of high-quality datasets.
*   **Spaces**: Be a part of our community by hosting your own projects and collaborating with fellow AI enthusiasts.

**Our Awesome Features**

*   **Inference Providers**: Tap into the power of top-notch inference providers like Groq, Cerebras, and SambaNova.
*   **Applications**: Build upon our latest applications, including vLLM, llama.cpp, MLX LM, and Ollama.
*   **Languages**: Get started with popular languages like PyTorch, TensorFlow, JAX, and more.

**Join the Party!**

Ready to join the machine learning revolution? Sign up for our platform today and discover a world of possibilities waiting for you!

**Careers at Hugging Face**

Want to be part of the magic? Check out our job openings and become one of our AI rockstars!

[Learn More](link)

We can't wait to welcome you on board!